# Модель эмоций из репо

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
drive_repo_path = "/content/drive/MyDrive/GigaAM_repo"
repo_path = "/content/GigaAM"
shutil.copytree(drive_repo_path, repo_path, dirs_exist_ok=True)
%cd {repo_path}
!pip install -e .

print("Библиотека восстановлена")

/content/GigaAM
Obtaining file:///content/GigaAM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 14.5 MB/s eta 0:00:00
  Building editable for gig

In [3]:
import gigaam
import time

print("Загружаем GigaAM-Emo...")
start_load = time.time()
model_emo = gigaam.load_model("emo").float()
print(f"Загружена за {time.time() - start_load:.2f} сек")

Загружаем GigaAM-Emo...


100%|███████████████████████████████████████| 462M/462M [00:07<00:00, 66.4MiB/s]


Загружена за 9.39 сек


Для обучения

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

old_weight = model_emo.head.weight.data.clone()
old_bias = model_emo.head.bias.data.clone()

new_head = nn.Linear(768, 6).to(device)

with torch.no_grad():
    new_head.weight.data[:4, :] = old_weight
    new_head.bias.data[:4] = old_bias

    nn.init.normal_(new_head.weight.data[4:], mean=0.0, std=1e-3)
    nn.init.zeros_(new_head.bias.data[4:])

model_emo.head = new_head

for name, param in model_emo.named_parameters():
    if 'head' in name:
        param.requires_grad = True
    elif 'layers' in name:
        layer_num = int(name.split('.')[2])
        if layer_num >= 14:
            param.requires_grad = True
    else:
        param.requires_grad = False

emotions = ['absent', 'anger', 'joy', 'neutral', 'sadness', 'fear']
emotion2id = {emotion: i for i, emotion in enumerate(emotions)}
id2emotion = {i: emotion for emotion, i in emotion2id.items()}

In [5]:
# Загружаем потом (сначала создаём такую же модель!)
#model_emo = gigaam.load_model("emo")
#model_emo.classifier = nn.Linear(768, 6)  # меняем последний слой
model_emo.load_state_dict(torch.load("/content/drive/MyDrive/emo_model_6classes_weights.pth"))

<All keys matched successfully>

In [5]:
!pip install audiomentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 22.4 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.0.0
    Uninstalling soxr-1.0.0:
      Successfully uninstalled soxr-1.0.0


In [6]:
import audiomentations as A
import soundfile as sf
import librosa
from pathlib import Path
import random

# Настройка аугментатора (такой же, как ты использовала)
augmenter = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.5),
    A.PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
    A.TimeStretch(min_rate=0.9, max_rate=1.1, p=0.5),
    A.Shift(min_shift=-0.05, max_shift=0.05, rollover=False, p=0.5),
])

# Исходная папка и новая папка для расширенного датасета
src_dir = Path("/content/drive/MyDrive/876/")
dst_dir = Path("/content/drive/MyDrive/876_augmented/")
dst_dir.mkdir(exist_ok=True)

# Количество аугментированных копий на каждый оригинал
NUM_AUGMENTED = 3

for wav_path in src_dir.glob("*.wav"):
    print(f"Обработка {wav_path.name}")

    # 1. Копируем оригинал в новую папку
    original_dst = dst_dir / wav_path.name
    sf.write(original_dst, librosa.load(wav_path, sr=16000)[0], 16000)

    # 2. Загружаем аудио для аугментации
    audio, sr = librosa.load(wav_path, sr=16000)

    # 3. Создаём несколько аугментированных копий
    for i in range(NUM_AUGMENTED):
        augmented_audio = augmenter(samples=audio, sample_rate=sr)
        aug_path = dst_dir / f"{wav_path.stem}_aug{i}{wav_path.suffix}"
        sf.write(aug_path, augmented_audio, sr)

print(f"Датасет расширен. Файлы сохранены в {dst_dir}")

Обработка command_sadness_neutral_001.wav
Обработка statement_sadness_neutral_006.wav
Обработка statement_sadness_neutral_003.wav
Обработка statement_sadness_neutral_002.wav
Обработка other_sadness_neutral_001.wav
Обработка command_neutral_neutral_008.wav
Обработка statement_sadness_neutral_004.wav
Обработка question_sadness_neutral_003.wav
Обработка statement_sadness_neutral_005.wav
Обработка statement_sadness_neutral_009.wav
Обработка statement_sadness_neutral_011.wav
Обработка statement_sadness_neutral_013.wav
Обработка statement_sadness_neutral_014.wav
Обработка statement_sadness_neutral_015.wav
Обработка statement_sadness_neutral_012.wav
Обработка command_sadness_neutral_002.wav
Обработка statement_joy_neutral_022.wav
Обработка question_anger_warning_054.wav
Обработка other_sadness_neutral_002.wav
Обработка statement_sadness_neutral_007.wav
Обработка statement_sadness_neutral_016.wav
Обработка command_sadness_neutral_003.wav
Обработка command_sadness_neutral_004.wav
Обработка stat

In [7]:
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import librosa
import soundfile as sf
import tempfile
import random

class FolderEmotionDataset(Dataset):
    def __init__(self, folder_path=None, paths=None, labels=None):
        self.files = []

        if folder_path is not None:
            for f in Path(folder_path).iterdir():
                if f.suffix in ['.wav', '.mp3']:
                    emotion = f.stem.split('_')[1]
                    if emotion in emotion2id:
                        self.files.append((str(f), emotion2id[emotion]))
        elif paths is not None and labels is not None:
            self.files = list(zip(paths, labels))
        else:
            raise ValueError("Нужно указать либо folder_path, либо paths и labels")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        return path, label

    @classmethod
    def from_lists(cls, paths, labels):
        return cls(paths=paths, labels=labels)

def collate_fn(batch):
    paths, labels = zip(*batch)
    return list(paths), torch.tensor(labels)

dataset = FolderEmotionDataset(folder_path="/content/drive/MyDrive/876_augmented/")

In [8]:
from sklearn.model_selection import train_test_split

all_paths = [item[0] for item in dataset.files]
all_labels = [item[1] for item in dataset.files]

train_paths, val_test_paths, train_labels, val_test_labels = train_test_split(
    all_paths, all_labels, test_size=0.4, random_state=42, stratify=all_labels
)

test_paths, val_paths, test_labels, val_labels = train_test_split(
    val_test_paths, val_test_labels, test_size=0.5, random_state=42, stratify=val_test_labels
)

train_dataset = FolderEmotionDataset.from_lists(paths=train_paths, labels=train_labels)
val_dataset = FolderEmotionDataset.from_lists(paths=val_paths, labels=val_labels)
test_dataset = FolderEmotionDataset.from_lists(paths=test_paths, labels=test_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Все метки из тренировочного датасета
all_train_labels = [item[1] for item in train_dataset.files]

# Вычисляем веса
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(all_train_labels),
    y=all_train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Подставляем в loss
criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_emo.parameters()),
                       lr=1e-4,
                       weight_decay=1e-4)

best_val_loss = float('inf')
best_model_state = None
model_emo.to(device)

for epoch in range(20):
    model_emo.train()
    train_loss = 0
    train_batches = 0

    for paths, labels in train_loader:
        optimizer.zero_grad()

        batch_logits = []
        valid_labels = []

        for i, path in enumerate(paths):
            logits = model_emo.get_logits(path)

            if torch.isnan(logits).any():
                print(f"⚠️ NaN в {path}, пропускаем")
                continue

            batch_logits.append(logits)
            valid_labels.append(labels[i])

        if len(batch_logits) == 0:
            continue

        batch_logits = torch.stack(batch_logits)
        valid_labels = torch.tensor(valid_labels).to(device)

        loss = criterion(batch_logits, valid_labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches if train_batches > 0 else 0

    model_emo.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for paths, labels in val_loader:
            batch_logits = []
            valid_labels = []

            for i, path in enumerate(paths):
                logits = model_emo.get_logits(path)

                if torch.isnan(logits).any():
                    print(f"⚠️ NaN в {path}, пропускаем")
                    continue

                batch_logits.append(logits)
                valid_labels.append(labels[i])

            if len(batch_logits) == 0:
                continue

            batch_logits = torch.stack(batch_logits)
            valid_labels = torch.tensor(valid_labels).to(device)
            loss = criterion(batch_logits, valid_labels)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches if val_batches > 0 else 0

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model_emo.state_dict().copy()
        print(f"Лучшая модель (Val Loss: {best_val_loss:.4f})")

    print(f"Эпоха {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Лучшая модель (Val Loss: 1.2120)
Эпоха 1 | Train Loss: 1.5748 | Val Loss: 1.2120
Лучшая модель (Val Loss: 1.0213)
Эпоха 2 | Train Loss: 0.8086 | Val Loss: 1.0213
Лучшая модель (Val Loss: 0.6778)
Эпоха 3 | Train Loss: 0.4579 | Val Loss: 0.6778
Лучшая модель (Val Loss: 0.5549)
Эпоха 4 | Train Loss: 0.2636 | Val Loss: 0.5549
Лучшая модель (Val Loss: 0.5395)
Эпоха 5 | Train Loss: 0.1282 | Val Loss: 0.5395
Лучшая модель (Val Loss: 0.4890)
Эпоха 6 | Train Loss: 0.1007 | Val Loss: 0.4890
Лучшая модель (Val Loss: 0.4825)
Эпоха 7 | Train Loss: 0.1121 | Val Loss: 0.4825
Эпоха 8 | Train Loss: 0.0849 | Val Loss: 0.7542
Эпоха 9 | Train Loss: 0.0815 | Val Loss: 0.7031
Эпоха 10 | Train Loss: 0.0961 | Val Loss: 0.5125
Эпоха 11 | Train Loss: 0.0627 | Val Loss: 0.7282
Эпоха 12 | Train Loss: 0.0198 | Val Loss: 0.4964
Лучшая модель (Val Loss: 0.3657)
Эпоха 13 | Train Loss: 0.1167 | Val Loss: 0.3657
Эпоха 14 | Train Loss: 0.1053 | Val Loss: 0.6616
Лучшая модель (Val Loss: 0.2920)
Эпоха 15 | Train Loss: 0.0

In [10]:
torch.save(best_model_state, "/content/drive/MyDrive/emo_model_6classes_weights.pth")

In [11]:
model_emo.load_state_dict(best_model_state)
model_emo.eval()

test_loss = 0
test_batches = 0
all_preds = []
all_labels_list = []

with torch.no_grad():
    for paths, labels in test_loader:
        batch_logits = []
        valid_labels = []

        for path, label in zip(paths, labels):
            logits = model_emo.get_logits(path)

            if torch.isnan(logits).any():
                print(f"⚠️ NaN в {path}, пропускаем")
                continue

            batch_logits.append(logits)
            valid_labels.append(label)

        if len(batch_logits) == 0:
            continue

        batch_logits = torch.stack(batch_logits)
        valid_labels = torch.tensor(valid_labels).to(device)
        loss = criterion(batch_logits, valid_labels)

        test_loss += loss.item()
        test_batches += 1

        preds = torch.argmax(batch_logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(valid_labels.cpu().numpy())

from sklearn.metrics import accuracy_score, f1_score, classification_report

test_accuracy = accuracy_score(all_labels_list, all_preds)
test_f1 = f1_score(all_labels_list, all_preds, average='weighted')

print(f"Test Loss: {test_loss / test_batches:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 (weighted): {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels_list, all_preds, target_names=emotions, labels=list(range(6))))

Test Loss: 0.2474
Test Accuracy: 0.9358
Test F1 (weighted): 0.9361

Classification Report:
              precision    recall  f1-score   support

      absent       0.90      1.00      0.95        93
       anger       0.99      0.92      0.95       267
         joy       0.95      0.91      0.93        67
     neutral       0.88      0.94      0.91       142
     sadness       0.97      0.88      0.92        64
        fear       0.87      0.99      0.92        68

    accuracy                           0.94       701
   macro avg       0.93      0.94      0.93       701
weighted avg       0.94      0.94      0.94       701



Добавить позже

Эксперименты

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_emo.parameters()), lr=1e-10)

for epoch in range(5):
    for paths, labels in dataloader:
        optimizer.zero_grad()

        batch_logits = []
        for path in paths:
            wav, length = model_emo.prepare_wav(path)
            encoded, _ = model_emo.forward(wav, length)
            encoded_pooled = nn.functional.avg_pool1d(
                encoded, kernel_size=encoded.shape[-1]
            ).squeeze(-1)

            logits = model_emo.head(encoded_pooled)[0]

            if torch.isnan(logits).any():
                print(f"⚠️ NaN в {path}, пропускаем файл")
                continue

            logits = torch.clamp(logits, min=-5, max=5)

            batch_logits.append(logits)

        batch_logits = torch.stack(batch_logits)
        loss = criterion(batch_logits, labels.to(device))

        loss = criterion(batch_logits + 1e-8, labels.to(device))

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 3.3747
Epoch 2, Loss: 3.5831
Epoch 3, Loss: 4.8081


KeyboardInterrupt: 

Проверки

In [ ]:
# Найди все файлы короче 0.5 сек
import librosa
from pathlib import Path

folder = Path("/content/drive/MyDrive/testMySystem/")
short_files = []

for f in folder.iterdir():
    if f.suffix in ['.wav', '.mp3']:
        duration = librosa.get_duration(filename=str(f))
        if duration < 0.5:
            short_files.append((f.name, duration))

print("Короткие файлы:")
for name, dur in short_files:
    print(f"  {name}: {dur:.2f} сек")

/tmp/ipykernel_3423/2652077301.py:10: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration = librosa.get_duration(filename=str(f))


Короткие файлы:


In [ ]:
# Веса (768 входов × 7 выходов)
print("=== ВЕСА HEAD ===")
print(f"head.weight.shape: {model_emo.head.weight.shape}")

# Статистика по весам
print(f"\nСтарые классы (0-3):")
print(f"  min: {model_emo.head.weight.data[:4].min():.4f}")
print(f"  max: {model_emo.head.weight.data[:4].max():.4f}")
print(f"  mean: {model_emo.head.weight.data[:4].mean():.4f}")

"""print(f"\nНовые классы (4-6):")
print(f"  min: {model_emo.head.weight.data[4:].min():.4f}")
print(f"  max: {model_emo.head.weight.data[4:].max():.4f}")
print(f"  mean: {model_emo.head.weight.data[4:].mean():.4f}")

# Первые несколько значений для примера
print(f"\nПервые 5 весов для класса 0:\n{model_emo.head.weight[0, :5]}")
print(f"\nПервые 5 весов для класса 4:\n{model_emo.head.weight[4, :5]}")"""

print(f"\n=== BIAS HEAD ===")
print(f"head.bias.shape: {model_emo.head.bias.shape}")
print(f"Старые классы bias: {model_emo.head.bias.data[:4]}")
print(f"Новые классы bias: {model_emo.head.bias.data[4:]}")

=== ВЕСА HEAD ===
head.weight.shape: torch.Size([4, 768])

Старые классы (0-3):
  min: -0.0416
  max: 0.0420
  mean: 0.0002

=== BIAS HEAD ===
head.bias.shape: torch.Size([4])
Старые классы bias: tensor([0.0231, 0.0204, 0.0240, 0.0022], device='cuda:0')
Новые классы bias: tensor([], device='cuda:0')


Для теста

In [ ]:
import time

def recognize_emotion(audio_path):
    start_time = time.time()
    probs = model_emo.get_probs(audio_path)
    main_emotion = max(probs, key=probs.get)
    confidence = probs[main_emotion]

    elapsed = time.time() - start_time

    return probs, main_emotion, confidence, elapsed

In [ ]:
audio_path = "/content/drive/MyDrive/testMySystem/"

if not os.path.exists(audio_path):
    print("Файлы не найдены")

In [ ]:
from pathlib import Path

folder = Path(audio_path)
for audio_file in folder.iterdir():

  probs, emotion, confidence, elapsed = recognize_emotion(audio_file)

  print("Файл: ", audio_file.name)
  print("Вероятности эмоций:")
  for emo, prob in probs.items():
      print(f"{emo:10}: {prob:.3f}")
  print(f"Основная эмоция: {emotion} (уверенность: {confidence:.1%})")
  print(f"Время: {elapsed:.3f} сек\n")

AttributeError: 'Tensor' object has no attribute 'get'

In [ ]:
# 1. Посмотреть общее количество слоёв
num_layers = len(model_emo.encoder.layers)  # или model_emo.conformer.layers
print(f'Всего слоёв в энкодере: {num_layers}')

# 2. Напечатать 10 последних имён слоёв, чтобы понять структуру
print("\nПоследние слои модели:")
for name, _ in list(model_emo.named_parameters())[-20:]:
    print(name)

# 3. Самый главный способ: посмотреть, какие параметры содержат 'layer'
print(f"\nДоступные слои для разморозки:")
for name, _ in model_emo.named_parameters():
    if 'layer' in name.lower():
        # пытаемся извлечь номер слоя
        parts = name.split('.')
        for p in parts:
            if p.isdigit():
                print(f'  - {name}')
                break

Всего слоёв в энкодере: 16

Последние слои модели:
encoder.layers.15.self_attn.pos_bias_v
encoder.layers.15.self_attn.linear_q.weight
encoder.layers.15.self_attn.linear_q.bias
encoder.layers.15.self_attn.linear_k.weight
encoder.layers.15.self_attn.linear_k.bias
encoder.layers.15.self_attn.linear_v.weight
encoder.layers.15.self_attn.linear_v.bias
encoder.layers.15.self_attn.linear_out.weight
encoder.layers.15.self_attn.linear_out.bias
encoder.layers.15.self_attn.linear_pos.weight
encoder.layers.15.norm_feed_forward2.weight
encoder.layers.15.norm_feed_forward2.bias
encoder.layers.15.feed_forward2.linear1.weight
encoder.layers.15.feed_forward2.linear1.bias
encoder.layers.15.feed_forward2.linear2.weight
encoder.layers.15.feed_forward2.linear2.bias
encoder.layers.15.norm_out.weight
encoder.layers.15.norm_out.bias
head.weight
head.bias

Доступные слои для разморозки:
  - encoder.layers.0.norm_feed_forward1.weight
  - encoder.layers.0.norm_feed_forward1.bias
  - encoder.layers.0.feed_forward1

In [ ]:
# Просто напечатай конфиг целиком
print(model_emo.cfg)

{'id2name': ['angry', 'sad', 'neutral', 'positive'], 'preprocessor': {'_target_': 'gigaam.preprocess.FeatureExtractor', 'sample_rate': 16000, 'features': 64}, 'encoder': {'_target_': 'gigaam.encoder.ConformerEncoder', 'feat_in': 64, 'n_layers': 16, 'd_model': 768, 'subsampling_factor': 4, 'ff_expansion_factor': 4, 'self_attention_model': 'rel_pos', 'pos_emb_max_len': 5000, 'n_heads': 16, 'conv_kernel_size': 31, 'flash_attn': False}, 'head': {'_target_': 'torch.nn.Linear', 'in_features': 768, 'out_features': 4}, 'model_name': 'emo', 'hashes': {'model': 'e7ab8a0ce0c41721e9c731d5abf9c01e'}}
